In [6]:
import numpy as np
import time

In [2]:
class Source:
    def __init__(self, Y=10000):
        self.Y = Y

    @staticmethod
    def _generate_directions_array(n):
        """Векторная генерация n изотропных направлений."""
        phi = np.random.uniform(0, 2 * np.pi, n)
        cos_theta = np.random.uniform(-1, 1, n)

        sin_theta = np.sqrt(1 - cos_theta**2)

        return np.column_stack((
            sin_theta * np.cos(phi),
            sin_theta * np.sin(phi),
            cos_theta
        ))

    def generate_directions_array(self, E):
        """Генерирует массив направлений всех фотонов события."""
        if E < 0:
            raise ValueError("Энергия не может быть отрицательной.")

        n_photons = np.random.poisson(self.Y * E)

        return self._generate_directions_array(n_photons)

    def generate_directions(self, E, batch_size=100):
        """Генератор, возвращающий направления порциями по batch_size."""
        if E < 0:
            raise ValueError("Энергия не может быть отрицательной.")
        if batch_size <= 0:
            raise ValueError("Размер порции должен быть положительным.")

        n_photons = np.random.poisson(self.Y * E)

        for start in range(0, n_photons, batch_size):
            n = min(batch_size, n_photons - start)
            yield self._generate_directions_array(n)

In [3]:
class Detector:
    def __init__(self, R=2, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a

        self.pm_positions = self.generate_positions()

    def generate_positions(self):
        """Векторная генерация положений фотоумножителей на сфере."""
        i = np.arange(self.N)

        golden_angle = np.pi * (3 - np.sqrt(5))

        z = 1 - 2 * (i + 0.5) / self.N
        theta = golden_angle * i
        rho = np.sqrt(1 - z**2)

        x = self.R * rho * np.cos(theta)
        y = self.R * rho * np.sin(theta)
        z = self.R * z

        return np.column_stack((x, y, z))

    def is_on_detector(self, point, tol=1e-6):
        return abs(np.linalg.norm(point) - self.R) < tol

    def belongs_to_pm(self, point):
        """Проверяет попадание одной точки на поверхность ФЭУ."""
        point = np.asarray(point)

        if not self.is_on_detector(point):
            return False

        cos_angles = self.pm_positions @ point / self.R**2
        return np.any(cos_angles >= np.cos(self.r / self.R))

    def find_photomultiplier(self, source, direction):
        """Скалярный метод: определяет ФЭУ для одного фотона."""
        source = np.asarray(source)
        direction = np.asarray(direction)

        dot = np.dot(source, direction)

        t = -dot + np.sqrt(
            dot**2 + self.R**2 - np.dot(source, source)
        )

        hit_point = source + t * direction

        cos_angles = self.pm_positions @ hit_point / self.R**2
        hits = np.where(cos_angles >= np.cos(self.r / self.R))[0]

        if len(hits) == 0:
            return None

        return hits[0]

    def propagate_photons(
        self, directions, source_position, attenuation_length=None
    ):
        """
        Векторно распространяет массив фотонов до поверхности сферы.

        Возвращает:
        hits — булев массив попаданий на ФЭУ;
        t — массив расстояний от источника до поверхности.
        """
        directions = np.asarray(directions, dtype=float)
        source_position = np.asarray(source_position, dtype=float)

        if directions.ndim != 2 or directions.shape[1] != 3:
            raise ValueError("Массив направлений должен иметь форму (N, 3).")

        if source_position.shape != (3,):
            raise ValueError("Положение источника должно иметь три координаты.")

        if np.dot(source_position, source_position) > self.R**2:
            raise ValueError("Источник находится за пределами детектора.")

        if attenuation_length is not None and attenuation_length <= 0:
            raise ValueError("Длина затухания должна быть положительной.")

        # Расстояния от источника до поверхности сферы
        dot = directions @ source_position

        t = -dot + np.sqrt(
            dot**2 + self.R**2 - np.dot(source_position, source_position)
        )

        # Точки пересечения всех фотонов с поверхностью сферы
        hit_points = source_position + t[:, None] * directions

        # Матрица косинусов углов между точками попадания и положениями ФЭУ
        cos_angles = (hit_points / self.R) @ (
            self.pm_positions / self.R
        ).T

        # Каждый фотон учитывается не более одного раза
        hits = np.any(
            cos_angles >= np.cos(self.r / self.R),
            axis=1
        )

        # Экспоненциальное затухание из задачи 6
        if attenuation_length is not None:
            survival_probability = np.exp(-t / attenuation_length)

            survived = (
                np.random.random(len(directions)) < survival_probability
            )

            hits &= survived

        return hits, t

    def generate_photoelectrons(self, photon_hits):
        """
        Вычисляет суммарное число фотоэлектронов по массиву попаданий.
        """
        if np.isscalar(photon_hits):
            n_hits = int(photon_hits)
        else:
            photon_hits = np.asarray(photon_hits, dtype=bool)

            if photon_hits.ndim != 1:
                raise ValueError("Массив попаданий должен быть одномерным.")

            n_hits = np.count_nonzero(photon_hits)

        if n_hits == 0:
            return 0

        # Для каждого попавшего фотона разыгрываем число фотоэлектронов
        photoelectrons = np.random.poisson(self.a, size=n_hits)

        return int(np.sum(photoelectrons))

    def simulate_event(
        self, source, E, source_position, attenuation_length=None
    ):
        """Полностью векторизованное моделирование одного события."""

        # Все фотоны события хранятся в одном массиве
        directions = source.generate_directions_array(E)

        # Распространение фотонов и определение попаданий
        hits, distances = self.propagate_photons(
            directions,
            source_position,
            attenuation_length=attenuation_length
        )

        # Расчёт суммарного числа фотоэлектронов
        return self.generate_photoelectrons(hits)

In [4]:
def run_simulations(
    detector, source, E, position,
    n_events=1000, attenuation_length=None
):
    results = np.empty(n_events, dtype=int)

    for i in range(n_events):
        results[i] = detector.simulate_event(
            source,
            E,
            position,
            attenuation_length=attenuation_length
        )

    return results

In [5]:
source = Source(Y=10000)
detector = Detector(R=2, N=40, r=0.1, a=0.5)

# Пример моделирования с затуханием
results = run_simulations(
    detector,
    source,
    E=1.0,
    position=np.array([0.0, 0.0, 0.0]),
    n_events=1000,
    attenuation_length=8.0
)

print(f"Средний заряд: {results.mean():.2f}")
print(f"Стандартное отклонение: {results.std(ddof=1):.2f}")

Средний заряд: 97.49
Стандартное отклонение: 11.79


In [7]:
def generate_directions_scalar(n):
    directions = np.empty((n, 3))

    for i in range(n):
        phi = np.random.uniform(0, 2 * np.pi)
        cos_theta = np.random.uniform(-1, 1)

        sin_theta = np.sqrt(1 - cos_theta**2)

        directions[i, 0] = sin_theta * np.cos(phi)
        directions[i, 1] = sin_theta * np.sin(phi)
        directions[i, 2] = cos_theta

    return directions

In [8]:
def propagate_photons_scalar(detector, directions, source_position):
    hits = np.zeros(len(directions), dtype=bool)

    source_position = np.asarray(source_position)
    cos_limit = np.cos(detector.r / detector.R)

    for i, direction in enumerate(directions):

        dot = np.dot(direction, source_position)

        t = -dot + np.sqrt(
            dot**2
            + detector.R**2
            - np.dot(source_position, source_position)
        )

        hit_point = source_position + t * direction

        cos_angles = (
            detector.pm_positions @ hit_point / detector.R**2
        )

        hits[i] = np.any(cos_angles >= cos_limit)

    return hits

In [9]:
def generate_photoelectrons_scalar(detector, hits):
    n_photoelectrons = 0

    for hit in hits:
        if hit:
            n_photoelectrons += np.random.poisson(detector.a)

    return n_photoelectrons

In [10]:
def measure_time(function, repeats=3):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        function()
        times.append(time.perf_counter() - start)

    return np.median(times)


# Параметры сравнения
n_benchmark = 10000
n_repeats = 3

position = np.array([0.0, 0.0, 0.0])

# Один и тот же массив направлений для сравнения распространения
directions = source._generate_directions_array(n_benchmark)


# 1. Генерация направлений
time_gen_numpy = measure_time(
    lambda: source._generate_directions_array(n_benchmark),
    n_repeats
)

time_gen_scalar = measure_time(
    lambda: generate_directions_scalar(n_benchmark),
    n_repeats
)


# 2. Распространение фотонов
hits_numpy, _ = detector.propagate_photons(
    directions, position
)

hits_scalar = propagate_photons_scalar(
    detector, directions, position
)

# Проверяем совпадение результатов
n_differences = np.count_nonzero(hits_numpy != hits_scalar)

time_prop_numpy = measure_time(
    lambda: detector.propagate_photons(directions, position),
    n_repeats
)

time_prop_scalar = measure_time(
    lambda: propagate_photons_scalar(detector, directions, position),
    n_repeats
)


# 3. Вычисление числа фотоэлектронов
time_pe_numpy = measure_time(
    lambda: detector.generate_photoelectrons(hits_numpy),
    n_repeats
)

time_pe_scalar = measure_time(
    lambda: generate_photoelectrons_scalar(detector, hits_numpy),
    n_repeats
)


# Вывод результатов
print(f"Количество фотонов: {n_benchmark}")
print(f"Количество попаданий на ФЭУ: {np.sum(hits_numpy)}")
print(f"Различий между реализациями распространения: {n_differences}")

print("\nВремя выполнения:")

print(
    f"Генерация:          NumPy = {time_gen_numpy:.6f} с, "
    f"циклы = {time_gen_scalar:.6f} с, "
    f"ускорение = {time_gen_scalar / time_gen_numpy:.2f} раз"
)

print(
    f"Распространение:    NumPy = {time_prop_numpy:.6f} с, "
    f"циклы = {time_prop_scalar:.6f} с, "
    f"ускорение = {time_prop_scalar / time_prop_numpy:.2f} раз"
)

print(
    f"Фотоэлектроны:      NumPy = {time_pe_numpy:.6f} с, "
    f"циклы = {time_pe_scalar:.6f} с, "
    f"ускорение = {time_pe_scalar / time_pe_numpy:.2f} раз"
)

Количество фотонов: 10000
Количество попаданий на ФЭУ: 223
Различий между реализациями распространения: 0

Время выполнения:
Генерация:          NumPy = 0.000395 с, циклы = 0.056272 с, ускорение = 142.43 раз
Распространение:    NumPy = 0.001020 с, циклы = 0.107088 с, ускорение = 104.98 раз
Фотоэлектроны:      NumPy = 0.000016 с, циклы = 0.000347 с, ускорение = 22.08 раз


Векторизация дала существенное ускорение во всех трёх операциях.